# Plain carbon / AISI 1010 / Taylor — propiedades del acero

Comparación **en función de la temperatura**, no del tiempo: la tabla suministrada
contiene valores a temperaturas específicas. Las curvas Taylor se calculan llamando
las funciones actuales de `Material` del simulador; no se ajustan nuevas correlaciones.

Fuente de los datos: captura proporcionada por el usuario, *Appendix A — Thermophysical
Properties of Matter*, filas **Plain carbon (Mn ≤ 1%, Si ≤ 0.1%)** y **AISI 1010**.
Los valores de 300 K proceden de las columnas de propiedades a 300 K; los demás de
las columnas de temperaturas variables. En cada columna, k está arriba y cp abajo.
La captura no identifica la edición del libro ni permite verificar el número completo
de página; no se atribuye una edición no confirmada.

Los puntos se unen con segmentos rectos como guía visual. Estos segmentos no son
correlaciones publicadas. El intervalo se limita a los datos disponibles: 300–1000 K.


## Settings

In [1]:
from pathlib import Path
import sys
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "heattransfer/models").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from heattransfer.models.material import Material

settings = {"samples": 500, "height": 550}
table_temperature_K = np.array([300, 400, 600, 800, 1000], dtype=float)
materials = {
    "Plain carbon": {
        "k": np.array([60.5, 56.7, 48.0, 39.2, 30.0]),
        "cp": np.array([434, 487, 559, 685, 1169], dtype=float),
        "color": "#d95f02", "symbol": "circle", "dash": "dash",
    },
    "AISI 1010": {
        "k": np.array([63.9, 58.7, 48.8, 39.2, 31.3]),
        "cp": np.array([434, 487, 559, 685, 1168], dtype=float),
        "color": "#2ca02c", "symbol": "diamond-open", "dash": "dot",
    },
}
steel = Material(conductivity=None, specific_heat=None)
temperature_K = np.linspace(300, 1000, settings["samples"])
taylor = {
    "k": np.array([steel.thermal_conductivity(T) for T in temperature_K]),
    "cp": np.array([steel.heat_capacity(T) for T in temperature_K]),
}
taylor_at_points = {
    "k": np.array([steel.thermal_conductivity(T) for T in table_temperature_K]),
    "cp": np.array([steel.heat_capacity(T) for T in table_temperature_K]),
}
layout = {
    "height": settings["height"], "template": "plotly_white",
    "font": {"family": "Arial", "size": 15}, "hovermode": "x unified",
    "legend": {"orientation": "h", "x": 0, "y": 1.12},
    "margin": {"t": 125, "b": 70, "l": 95, "r": 40},
    "xaxis": {"title": "Temperatura [K]", "range": [290, 1010]},
}


def property_figure(key, title, ylabel):
    figure = go.Figure()
    figure.add_trace(go.Scatter(x=temperature_K, y=taylor[key], mode="lines",
        name="Taylor — implementación del simulador", line={"color": "#1f77b4", "width": 3}))
    for name, values in materials.items():
        figure.add_trace(go.Scatter(x=table_temperature_K, y=values[key], mode="lines+markers",
            name=name, line={"color": values["color"], "width": 2, "dash": values["dash"]},
            marker={"symbol": values["symbol"], "size": 10, "line": {"width": 2}}))
    figure.update_layout(**layout, title={"text": title, "x": 0.5}, yaxis_title=ylabel)
    return figure


## Plot — Conductividad

In [2]:
conductivity_fig = property_figure("k", "Conductividad térmica del acero", "k [W/(m·K)]")
conductivity_fig.show()


Plotly figure

## Plot — Calor específico
Los valores tabulados de cp coinciden hasta 800 K; a 1000 K difieren en 1 J/(kg·K). Los símbolos abiertos permiten ver la superposición.

In [3]:
heat_capacity_fig = property_figure("cp", "Calor específico del acero", "cp [J/(kg·K)]")
heat_capacity_fig.show()


Plotly figure

## Plot — Diferencias en los puntos tabulados
Se muestra `100 × (Taylor − tabla) / tabla`. No se calculan errores entre puntos, donde la tabla no proporciona valores.

In [4]:
difference_fig = make_subplots(rows=1, cols=2,
                               subplot_titles=["Conductividad", "Calor específico"])
for col, key in enumerate(["k", "cp"], 1):
    for name, values in materials.items():
        difference_fig.add_trace(go.Scatter(
            x=table_temperature_K, y=100*(taylor_at_points[key]/values[key]-1),
            mode="lines+markers", name=name, legendgroup=name, showlegend=col==1,
            line={"color":values["color"],"dash":values["dash"]},
            marker={"symbol":values["symbol"],"size":9}),row=1,col=col)
    difference_fig.add_hline(y=0,line_color="#888888",line_dash="dot",row=1,col=col)
    difference_fig.update_xaxes(title_text="Temperatura [K]",row=1,col=col)
    difference_fig.update_yaxes(title_text="(Taylor − tabla) / tabla [%]",row=1,col=col)
difference_fig.update_layout(height=500,template="plotly_white",
                             font={"family":"Arial","size":14},
                             margin={"t":100,"b":70,"l":80,"r":30},
                             legend={"orientation":"h","y":1.2},
                             hovermode="x unified")
difference_fig.show()


Plotly figure

## Valores y lectura del resultado

In [5]:
for key, label in [("k", "Conductividad [W/(m·K)]"), ("cp", "Calor específico [J/(kg·K)]")]:
    print(label)
    print(f"{'T [K]':>8} {'Plain carbon':>15} {'AISI 1010':>15} {'Taylor':>15}")
    for i, T in enumerate(table_temperature_K):
        print(f"{T:8.0f} {materials['Plain carbon'][key][i]:15.3f} {materials['AISI 1010'][key][i]:15.3f} {taylor_at_points[key][i]:15.5f}")
    print()


Conductividad [W/(m·K)]
   T [K]    Plain carbon       AISI 1010          Taylor
     300          60.500          63.900        60.87230
     400          56.700          58.700        56.50230
     600          48.000          48.800        47.76230
     800          39.200          39.200        39.02230
    1000          30.000          31.300        30.28230

Calor específico [J/(kg·K)]
   T [K]    Plain carbon       AISI 1010          Taylor
     300         434.000         434.000       434.00001
     400         487.000         487.000       486.99992
     600         559.000         559.000       558.99948
     800         685.000         685.000       684.99856
    1000        1169.000        1168.000      1168.99700



El polinomio de calor específico de Taylor reproduce los cinco valores de **Plain carbon**
con diferencias de redondeo. La conductividad de Taylor es un ajuste lineal, por lo que
no pasa exactamente por todos los puntos. Esta coincidencia numérica aporta evidencia
sobre el conjunto de datos ajustado, aunque Taylor no identifica un grado concreto.
No implica que SPCC y AISI 1010 sean materiales intercambiables.

## Referencias

Taylor, S. W., & Wang, S. (2017). Modelling steel strip heating within an annealing furnace.
*Pacific Journal of Mathematics for Industry, 9*, Article 5.
https://doi.org/10.1186/s40736-017-0030-7

Ecuaciones 26 y 27: p. 4 del PDF. Datos Plain carbon y AISI 1010:
transcripción de la captura del apéndice proporcionada en esta conversación;
referencia bibliográfica completa pendiente de confirmar la edición del libro.
